# Fine-tune YOLO11s for 100-Food Real-Time Recognition

This notebook fine-tunes **YOLO11s classification** on the Nutrify **100-food dataset** and exports the `.pt` and `.onnx` files used by the real-time camera API. YOLO11s gives a better speed/accuracy balance than YOLO11l for CPU camera use.

## What you get
1. Auto-extract of your 100-food zip + automatic train / val / test split
2. Optional cross-check against a metadata CSV (not required for training)
3. **100-class classification** fine-tune (`yolo11s-cls.pt`) for folder-only data
4. Validation metrics and test-image checks
5. ONNX export with a 100-output check
6. Real-time webcam, video, and speed tests

## Speed / accuracy notes
- `s` is small enough for CPU real-time use and still has good accuracy. Use `yolo11n-cls.pt` if your target device is too slow.
- Train on a **T4 GPU** in Colab. CPU training for 100 classes will be very slow.
- Lower `imgsz` = faster but less accurate (224 is enough for classification).
- Regularization is enabled below (`dropout`, `erasing`, flip + scale augmentation).

In [ ]:
# Install dependencies (runs on first execution)
%pip install -U ultralytics opencv-python

import ultralytics
print("Ultralytics", ultralytics.__version__)

## 1. Dataset: your 100-food zip from Google Cloud

**Recommended (no truncation, no 780 MB browser upload):** download the zip **directly from Google Cloud Storage** using the next cell (your service-account JSON + bucket name).

Alternative: browser upload of `/content/100_whole_foods.zip` (must reach 100% and the file list must stop changing).

Expected structure inside the zip (one folder per food):

```
100_whole_foods.zip
├── almonds/
├── apple/
├── apricot_fruit/
├── ... 96 more food folders ...
└── zucchini/
```

Only images are needed — no metadata CSV required.

In [ ]:
# Main 100-food training config
import random
import shutil
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path("/content") if IN_COLAB else Path.cwd()
if not IN_COLAB and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ZIP_PATH = Path("/content/100_whole_foods.zip") if IN_COLAB else PROJECT_ROOT / "100_whole_foods.zip"
DATA_DIR = Path("/content/100_foods_raw") if IN_COLAB else PROJECT_ROOT / "data" / "100_foods"
WORK_DIR = Path("/content/100_foods_split") if IN_COLAB else PROJECT_ROOT / "data" / "100_foods_split"
METADATA_CSV = PROJECT_ROOT / "data" / "100_foods_metada_updated.csv"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
EXPECTED_CLASS_COUNT = 100
SEED = 601
VAL_FRAC = 0.15
TEST_FRAC = 0.15
BASE_MODEL = "yolo11s-cls.pt"
IMAGE_SIZE = 224

print("Dataset:", DATA_DIR)
print("Split output:", WORK_DIR)
print("Base model:", BASE_MODEL)

In [ ]:
# ── Download the zip directly from Google Cloud Storage ──
# 1) Upload your service-account JSON to Colab, e.g. /content/google-storage-creds.json
#    (it's in the project at config/google-storage-creds.json)
# 2) Set BUCKET_NAME and BLOB_NAME below, then run this cell.

%pip install -q google-cloud-storage

from pathlib import Path
import os

BUCKET_NAME = "food-vision-project-images"   # ← your GCS bucket
BLOB_NAME = "100_whole_foods.zip"            # path of the zip inside the bucket
CREDS_JSON = Path("/content/google-storage-creds.json")
DEST = ZIP_PATH

if CREDS_JSON.exists():
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(CREDS_JSON)
    from google.cloud import storage

    client = storage.Client()
    blob = client.bucket(BUCKET_NAME).blob(BLOB_NAME)
    print(f"⬇️ Downloading gs://{BUCKET_NAME}/{BLOB_NAME} → {DEST} ...")
    blob.download_to_filename(str(DEST))
    print(f"✅ Downloaded {DEST.stat().st_size / 1e6:.1f} MB")
else:
    print("⚠️ Credentials JSON not found at /content/google-storage-creds.json")
    print("   Option A: upload the JSON from config/ and re-run.")
    print("   Option B: in the GCS console → object menu → 'Authenticated URL', then:")
    print('   !wget "<AUTHENTICATED_URL>" -O /content/100_whole_foods.zip')
    print("   Option C: make the object public temporarily, then:")
    print(f'   !wget "https://storage.googleapis.com/{BUCKET_NAME}/{BLOB_NAME}" -O /content/100_whole_foods.zip')

In [ ]:
# ── Step 0: find and extract the uploaded zip ──
if ZIP_PATH and not ZIP_PATH.exists():
    # Maybe the uploaded file has a different name — find any .zip in /content
    candidates = sorted(Path("/content").glob("*.zip")) if IN_COLAB else []
    if candidates:
        ZIP_PATH = candidates[0]
        print("ℹ️ Using the zip file found at:", ZIP_PATH)

if ZIP_PATH and ZIP_PATH.exists():
    import time as _t

    # Wait if the upload is still in progress (file size keeps changing)
    stable = False
    for _ in range(6):
        s1 = ZIP_PATH.stat().st_size
        _t.sleep(3)
        s2 = ZIP_PATH.stat().st_size
        if s1 == s2:
            stable = True
            break
    size_mb = ZIP_PATH.stat().st_size / 1e6
    print(f"Zip size: {size_mb:.1f} MB —", "stable ✅" if stable else "STILL CHANGING ⚠️")

    # Make sure it really is a complete zip before extracting
    if not zipfile.is_zipfile(ZIP_PATH):
        head = ZIP_PATH.read_bytes()[:4]
        if head == b"PK\x03\x04":
            reason = (
                "The file STARTS like a real zip, but its END "
                "(central directory) is missing or unreadable.\n"
                "→ This almost always means the upload was interrupted / truncated.\n"
                "→ In Colab's Files panel: delete it, upload the zip again, and WAIT "
                "until the upload reaches 100% before re-running this cell."
            )
        else:
            reason = (
                "Wrong file uploaded (.rar / .7z are NOT zip files), "
                "or you downloaded an HTML page from Google Drive instead of the file, "
                "or you uploaded a folder (right-click → Compress to ZIP first)."
            )
        raise RuntimeError(
            f"❌ {ZIP_PATH} is NOT a valid zip file ({size_mb:.1f} MB).\n" + reason
        )

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print("📦 Extracted zip →", DATA_DIR)
    # If the zip wrapped everything in a single top folder, use that folder
    children = [p for p in DATA_DIR.iterdir() if p.is_dir()]
    has_loose_images = any(f.suffix.lower() in IMG_EXTS for f in DATA_DIR.iterdir() if f.is_file())
    if len(children) == 1 and not has_loose_images:
        DATA_DIR = children[0]
        print("   (zip contained a top folder — using:", DATA_DIR, ")")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATA_DIR}\n"
        "Upload your 100-food zip (set ZIP_PATH) or place the folders there, then re-run."
    )
print("Dataset found:", DATA_DIR)

# 1) Find classes: any folder that DIRECTLY contains images counts.
#    Handles BOTH layouts:
#      • flat:      apple/xxx.jpg
#      • split:     train/apple/xxx.jpg  +  test/apple/yyy.jpg  (merged per class)
from collections import defaultdict

class_images = defaultdict(list)
for folder in DATA_DIR.rglob("*"):
    if not folder.is_dir():
        continue
    imgs = [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS]
    if imgs:
        class_images[folder.name].extend(imgs)

classes = sorted(class_images.keys())
print(f"Found {len(classes)} classes:")
for c in classes:
    print(f"  {c:<18} {len(class_images[c]):>4} images")

if not classes:
    raise RuntimeError(
        f"No class folders found in {DATA_DIR}.\n"
        "Check the zip structure — it should be:  <food_name>/<images>.jpg\n"
        "(e.g. apple/photo1.jpg) — or train/<food>/ + test/<food>/."
    )

if len(classes) != EXPECTED_CLASS_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_CLASS_COUNT} food folders, but found {len(classes)}. "
        "Check that the 100-food zip is complete and has one folder per class."
    )

# 2) Rebuild a deterministic split with no image shared between splits.
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

for c in classes:
    imgs = sorted(class_images[c])
    random.Random(f"{SEED}:{c}").shuffle(imgs)
    val_count = max(1, int(len(imgs) * VAL_FRAC))
    test_count = max(1, int(len(imgs) * TEST_FRAC))
    if len(imgs) - val_count - test_count < 1:
        raise RuntimeError(f"Class {c} needs at least 3 images, found {len(imgs)}")

    split_images = {
        "val": imgs[:val_count],
        "test": imgs[val_count:val_count + test_count],
        "train": imgs[val_count + test_count:],
    }
    for split, selected in split_images.items():
        dst = WORK_DIR / split / c
        dst.mkdir(parents=True, exist_ok=True)
        for index, src in enumerate(selected):
            shutil.copy2(src, dst / f"{index:04d}_{src.name}")

print("\nSplit complete:")
for split in ("train", "val", "test"):
    sdir = WORK_DIR / split
    if sdir.exists():
        n = sum(1 for p in sdir.rglob("*") if p.suffix.lower() in IMG_EXTS)
        print(f"  {split:<6} {n:>4} images")

## 2. Cross-check against the metadata CSV (OPTIONAL — skip if you don't have one)

The metadata CSV is **not used for training** — YOLO only needs the images in their class folders. This cell is just an audit: it checks that every image has a metadata row (source URL, dimensions, download time).

The cell **auto-finds** any `.csv` inside the extracted dataset folder (or `/content`), so you don't need to set the path manually.

In [ ]:
import csv
from pathlib import Path

# Auto-find the metadata CSV inside the extracted dataset or /content
if not METADATA_CSV.exists():
    candidates = sorted(DATA_DIR.rglob("*.csv"))
    if not candidates:
        candidates = sorted(Path("/content").glob("*.csv"))
    if candidates:
        METADATA_CSV = candidates[0]
        print("ℹ️ Using metadata CSV found at:", METADATA_CSV)

if METADATA_CSV.exists():
    with METADATA_CSV.open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"Metadata rows: {len(rows)}")
    print("Columns:", list(rows[0].keys()) if rows else "<empty>")

    # images on disk vs rows in metadata
    disk = {p.name for p in DATA_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS}

    # try to match by image filename or image_id column
    id_col = "image_id" if rows and "image_id" in rows[0] else None
    path_col = "img_url" if rows and "img_url" in rows[0] else None
    meta_ids = {r[id_col] for r in rows} if id_col else set()
    meta_names = {Path(r[path_col]).name for r in rows if r.get(path_col)} if path_col else set()

    missing_from_meta = disk - meta_ids - meta_names
    print(f"Images on disk: {len(disk)}")
    print(f"Images missing from metadata: {len(missing_from_meta)}")
    if missing_from_meta:
        print("  Examples:", list(sorted(missing_from_meta))[:5])
else:
    print("⚠️ No metadata CSV found anywhere.")
    print("   Training doesn't need it — you can skip this cell and continue.")

## 3. Choose your mode

| Mode | Weights | Output | When to use |
|---|---|---|---|
| **Classification** recommended | `yolo11s-cls.pt` | one of 100 food names + confidence | Current folder-only data and real-time camera use |
| **Detection** | `yolo11s.pt` | food name + bounding box | Only if you later add box annotations |

For the Nutrify app (single food per photo, nutrition lookup), **classification is enough** and needs zero extra labeling.

In [ ]:
# ── Write classification dataset YAML ──
from pathlib import Path

try:
    DATA_DIR
except NameError:
    raise RuntimeError(
        "Run the config cell first (Step 1) — it defines DATA_DIR and builds the split."
    )

try:
    classes
except NameError:
    from collections import defaultdict
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    class_images = defaultdict(list)
    for folder in DATA_DIR.rglob("*"):
        if folder.is_dir():
            imgs = [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS]
            if imgs:
                class_images[folder.name].extend(imgs)
    classes = sorted(class_images.keys())
    print("ℹ️ Re-discovered", len(classes), "classes:", classes)

if not classes:
    raise RuntimeError(
        f"No class folders found in {DATA_DIR}.\n"
        "The extracted dataset is missing (Colab wipes /content on restart), or the zip "
        "has an unexpected structure (should be <food_name>/<images>.jpg).\n"
        "Re-run in order:  GCS download cell → config cell (extract + split) → this cell."
    )

WORK_DIR.mkdir(parents=True, exist_ok=True)
yaml_cls = WORK_DIR / "food100_cls.yaml"
lines = [
    f"path: {WORK_DIR.resolve().as_posix()}",
    "train: train",
    "val: val",
]
if (WORK_DIR / "test").exists():
    lines.append("test: test")
lines.append("names:")
for i, c in enumerate(classes):
    lines.append(f"  {i}: {c}")
yaml_cls.write_text("\n".join(lines), encoding="utf-8")
print(yaml_cls.read_text(encoding="utf-8"))

## 4. Fine-tune (classification)

Key settings:
- `data=WORK_DIR` — Ultralytics classification training takes a **directory** containing `train/val`, not a YAML. The YAML cell above is optional (you can skip it).
- `epochs=80` — gives the 100 classes time to learn. Early stopping cuts training short if validation stops improving.
- `imgsz=224` — YOLO11 classification default; increase to 320 if accuracy stalls, at the cost of speed.
- `batch=32` — fits a YOLO11s classifier on a T4 at 224×224; lower it if memory runs out.
- `dropout=0.2` + `erasing=0.4` — helps reduce overfitting.
- `device=0` for NVIDIA GPU (Colab T4), otherwise `cpu` (very slow).

**🛡 Crash-proof training:**
- Checkpoints are saved **every epoch** (`last.pt`, `best.pt`, numbered checkpoints) — to **Google Drive** when running in Colab.
- If Colab disconnects, the internet drops, or the power goes out: **just re-run the cells**. Training **resumes from the last completed epoch**, never from scratch.
- Every epoch prints its result plus elapsed time and an **estimated time remaining**.

## 4a. Checkpoint status & Google Drive mount

Run this cell **every time you (re)start the notebook**:
- Mounts Google Drive (in Colab) so checkpoints survive disconnects and runtime recycling.
- Prints how many epochs are already done and the estimated time remaining.
- The training cell below then **resumes automatically** instead of starting over.

In [ ]:
import csv
from pathlib import Path

# ── Google Drive mount (Colab only) so checkpoints survive disconnects ──
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/nutrify_yolo_food100")
    print("✅ Checkpoints will be saved to Google Drive")
except ImportError:
    PROJECT_DIR = Path("runs")
    print("ℹ️ Not in Colab — checkpoints saved locally under runs/")

EPOCHS = 80
RUN_NAME = "cls_train"
# NOTE: must equal Ultralytics' save dir = project/name (no extra folders!)
RUN_DIR = PROJECT_DIR / RUN_NAME

# ── Checkpoint status: where did we stop last time? ──
last_ckpt = RUN_DIR / "weights" / "last.pt"
results_csv = RUN_DIR / "results.csv"

In [ ]:
import time
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print("Training on:", "GPU (CUDA)" if device == 0 else "CPU")

RUN_DIR.mkdir(parents=True, exist_ok=True)
last_ckpt = RUN_DIR / "weights" / "last.pt"

# Resume from the last checkpoint if a previous run was interrupted
if last_ckpt.exists():
    model = YOLO(str(last_ckpt))
    print("⏯️ Resuming from checkpoint:", last_ckpt)
else:
    model = YOLO(BASE_MODEL)
    print("Starting fresh training with", BASE_MODEL)

# ── Progress tracker: logs every epoch + estimates time remaining ──
class EpochTracker:
    def __init__(self):
        self.start = time.time()

    def on_fit_epoch_end(self, trainer):
        n = trainer.epoch + 1                          # epoch just finished
        total = trainer.start_epoch + trainer.epochs   # overall total
        elapsed = time.time() - self.start
        per_epoch = elapsed / max(1, n - trainer.start_epoch)
        eta_min = per_epoch * max(0, total - n) / 60
        top1 = None
        try:
            top1 = trainer.metrics.get("metrics/accuracy_top1")
        except Exception:
            pass
        top1_s = f"top1={top1:.4f}" if top1 is not None else "top1=n/a"
        print(f"\n⏱ [EPOCH {n}/{total}] {top1_s} | "
              f"elapsed {elapsed/60:.1f} min | remaining ≈ {eta_min:.1f} min")

model.add_callback("on_fit_epoch_end", EpochTracker().on_fit_epoch_end)

results = model.train(
    data=str(WORK_DIR),   # classify needs a directory (train/val inside), not a YAML
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=32,
    lr0=0.01,
    patience=15,
    dropout=0.2,
    erasing=0.4,
    device=device,
    seed=SEED,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    resume=last_ckpt.exists(),   # ✅ continue from where it stopped
    save_period=1,               # ✅ checkpoint file every epoch
    fliplr=0.5,
    scale=0.5,
    verbose=True,
)
print("Best weights:", model.trainer.best)

## 5. Validate on the test split

Reports **top-1 / top-5 accuracy** per class and the confusion matrix.

In [ ]:
best = RUN_DIR / "weights" / "best.pt"
if best.exists():
    val_model = YOLO(str(best))
    metrics = val_model.val(
        data=str(WORK_DIR),   # directory, not YAML
        split="test" if (WORK_DIR / "test").exists() else "val",
    )
    print("Top-1 accuracy:", metrics.top1)
    print("Top-5 accuracy:", metrics.top5)
else:
    print("Run the training cell first.")

In [ ]:
# App-style inference over all 100 food classes.
# Keep the folder names unchanged because the backend uses the same sorted list.
def predict_app_food(img_path):
    res = val_model.predict(str(img_path), imgsz=224, verbose=False)[0]
    if res.probs is None:
        return None, 0.0
    model_classes = [res.names[i] for i in range(len(res.names))]
    if model_classes != classes:
        raise RuntimeError("Model class order does not match the 100-food folder order")
    top_index = int(res.probs.top1)
    return res.names[top_index], float(res.probs.top1conf)

# Example:
# food, conf = predict_app_food("path/to/photo.jpg")
# print("Detected:", food, "| confidence:", round(conf, 3))

## 6. Export for deployment

ONNX runs in the FastAPI backend without PyTorch. This cell writes the exact deployment names used in `model/`, saves the class order, and checks that ONNX returns 100 scores.

In [ ]:
import shutil
import numpy as np
%pip install -q onnxruntime
import onnxruntime as ort

best = RUN_DIR / "weights" / "best.pt"
if not best.exists():
    raise FileNotFoundError("Run the training cell first; best.pt was not found.")

exported = Path(YOLO(str(best)).export(
    format="onnx",
    imgsz=IMAGE_SIZE,
    dynamic=True,
    simplify=True,
))

DEPLOY_DIR = PROJECT_DIR / "deploy"
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
deploy_pt = DEPLOY_DIR / "realtime_food_recognition_100_foods.pt"
deploy_onnx = DEPLOY_DIR / "realtime_food_recognition_100_foods.onnx"
deploy_classes = DEPLOY_DIR / "realtime_food_recognition_100_foods_classes.txt"
shutil.copy2(best, deploy_pt)
shutil.copy2(exported, deploy_onnx)
deploy_classes.write_text("\n".join(classes) + "\n", encoding="utf-8")

session = ort.InferenceSession(str(deploy_onnx), providers=["CPUExecutionProvider"])
input_info = session.get_inputs()[0]
dummy = np.zeros((1, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
scores = session.run(None, {input_info.name: dummy})[0]
if scores.ndim != 2 or scores.shape[1] != EXPECTED_CLASS_COUNT:
    raise RuntimeError(f"Expected ONNX output [batch, 100], received {scores.shape}")

print("Deployment files are ready:")
print(" ", deploy_pt)
print(" ", deploy_onnx)
print(" ", deploy_classes)
print("Copy the .pt and .onnx files into the project's model/ folder.")

## 7. Real-time webcam test 🎥

Runs the trained model on live webcam frames (press `q` to quit). This is the exact loop you'd move into the app's live-scan mode.

In [ ]:
import cv2

# NOTE: Colab has no webcam — run this cell on your local machine
# after copying the trained weights folder from Drive.
rt_model = YOLO(str(RUN_DIR / "weights" / "best.pt"))
cap = cv2.VideoCapture(0)  # 0 = default webcam; change if needed

print("Live detection started — press 'q' to quit")
while cap.isOpened():
    ok, frame = cap.read()
    if not ok:
        break

    # Run inference (classification: top1 label + confidence)
    results = rt_model.predict(frame, imgsz=224, conf=0.25, verbose=False)
    r = results[0]
    if r.probs is not None:
        label = f"{r.names[int(r.probs.top1)]}  {float(r.probs.top1conf):.2f}"
    else:
        label = "..."

    cv2.putText(frame, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.imshow("Nutrify - Live Food Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# ── Test the trained model on a single image (works in Colab) ──
# Colab has no webcam/display, so use this instead of the cell above.
import random
import cv2 as _cv

rt_model = YOLO(str(RUN_DIR / "weights" / "best.pt"))

# Pick a random image from the test split
imgs = [p for p in (WORK_DIR / "test").rglob("*") if p.suffix.lower() in IMG_EXTS]
img = random.choice(imgs)
print("Testing on:", img, "(true class:", img.parent.name + ")")

res = rt_model.predict(str(img), imgsz=224, verbose=False)[0]
top1 = res.names[int(res.probs.top1)] if res.probs is not None else "?"
conf = float(res.probs.top1conf) if res.probs is not None else 0.0
print(f"Predicted: {top1}  ({conf:.3f})")

# Display the image with the prediction drawn on it
frame = _cv.imread(str(img))
frame = _cv.resize(frame, (420, 420))
_cv.putText(frame, f"{top1} {conf:.2f}", (10, 30),
            _cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

try:
    from google.colab.patches import cv2_imshow
    cv2_imshow(frame)          # Colab inline display
except ImportError:
    _cv.imshow("Nutrify - Prediction", frame)   # local window
    _cv.waitKey(0)
    _cv.destroyAllWindows()

In [ ]:
# ── Real-time food detection test 🎥 ──
# Model locations:
#   PyTorch: /content/drive/MyDrive/nutrify_yolo_food100/deploy/realtime_food_recognition_100_foods.pt
#   ONNX   : /content/drive/MyDrive/nutrify_yolo_food100/deploy/realtime_food_recognition_100_foods.onnx
#
# Three modes (auto-picked):
#   • Webcam        → only works on YOUR local machine, never in Colab
#   • Video file    → upload an .mp4 to Colab and set VIDEO_PATH below
#   • Image speed test → fallback (measures FPS on test images)

import time as _t
import random
import torch
import cv2 as _cv
from pathlib import Path

# Use the GPU if this Colab runtime has one (Runtime → Change runtime type → T4)
DEV = 0 if torch.cuda.is_available() else "cpu"
print("Inference device:", "GPU (CUDA)" if DEV == 0 else "CPU ⚠️ (slow)")

MODEL_PT = Path("/content/drive/MyDrive/nutrify_yolo_food100/deploy/realtime_food_recognition_100_foods.pt")
MODEL_ONNX = Path("/content/drive/MyDrive/nutrify_yolo_food100/deploy/realtime_food_recognition_100_foods.onnx")
VIDEO_PATH = Path("/content/test_video.mp4")   # ← your uploaded video (Colab)

if MODEL_PT.exists():
    rt_model = YOLO(str(MODEL_PT))
    print("✅ Loaded PyTorch model:", MODEL_PT)
elif MODEL_ONNX.exists():
    rt_model = YOLO(str(MODEL_ONNX))
    print("✅ Loaded ONNX model:", MODEL_ONNX)
else:
    raise FileNotFoundError("Model not found — check the best.pt / best.onnx paths above.")

if DEV == 0:
    rt_model.to(0)   # move weights to GPU

# Warm-up (first inference includes model setup — don't time it)
_warm = [p for p in (WORK_DIR / "test").rglob("*") if p.suffix.lower() in IMG_EXTS][0]
rt_model.predict(_cv.imread(str(_warm)), imgsz=224, device=DEV, verbose=False)

cap = _cv.VideoCapture(0)
if cap.isOpened():
    # ── Live webcam mode (local machine only) ──
    print("Live detection started — press 'q' to quit")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = _t.time()
        res = rt_model.predict(frame, imgsz=224, device=DEV, verbose=False)[0]
        fps = 1.0 / max(_t.time() - t0, 1e-6)
        top1 = res.names[int(res.probs.top1)] if res.probs is not None else "?"
        conf = float(res.probs.top1conf) if res.probs is not None else 0.0
        _cv.putText(frame, f"{top1} {conf:.2f} | {fps:.1f} FPS", (10, 30),
                    _cv.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        _cv.imshow("Nutrify - Real-time Food Detection", frame)
        if _cv.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    _cv.destroyAllWindows()
elif VIDEO_PATH.exists():
    # ── Video-file mode (works in Colab — upload an .mp4) ──
    vcap = _cv.VideoCapture(str(VIDEO_PATH))
    print("🎬 Video mode:", VIDEO_PATH.name)
    n = 0
    t0 = _t.time()
    seen = []
    while vcap.isOpened():
        ok, frame = vcap.read()
        if not ok:
            break
        res = rt_model.predict(frame, imgsz=224, device=DEV, verbose=False)[0]
        top1 = res.names[int(res.probs.top1)] if res.probs is not None else "?"
        conf = float(res.probs.top1conf) if res.probs is not None else 0.0
        if n % 30 == 0:
            seen.append((n, top1, conf))
        n += 1
    dt = _t.time() - t0
    vcap.release()
    print(f"Processed {n} frames in {dt:.1f}s → {n/dt:.1f} FPS (including video decode)")
    for s in seen:
        print(f"  frame {s[0]:>4}: {s[1]:<14} {s[2]:.2f}")
else:
    # ── Colab / no camera / no video: speed test over test images ──
    print("No webcam or video available — measuring real-time speed on test images instead.\n")
    imgs = [p for p in (WORK_DIR / "test").rglob("*") if p.suffix.lower() in IMG_EXTS]
    random.shuffle(imgs)
    times = []
    for img in imgs[:30]:
        frame = _cv.imread(str(img))
        t0 = _t.time()
        res = rt_model.predict(frame, imgsz=224, device=DEV, verbose=False)[0]
        dt = _t.time() - t0
        times.append(dt)
        top1 = res.names[int(res.probs.top1)] if res.probs is not None else "?"
        conf = float(res.probs.top1conf) if res.probs is not None else 0.0
        print(f"  {img.parent.name:<14} → {top1:<14} {conf:.2f}   ({1/dt:.1f} FPS)")
    avg = sum(times) / len(times)
    print(f"\nAverage: {avg*1000:.1f} ms/frame → {1/avg:.1f} FPS on this machine")

## 8. (Optional) Detection mode with bounding boxes

Only worth it if you have **real bounding-box annotations**. If you don't, the cell below fakes boxes by using the whole image — the model will learn "the whole picture is the food", which is weak localization but can still work for full-frame food photos.

Better long-term: label a subset with [Roboflow](https://roboflow.com) or use a public food-detection dataset (UECFOOD-100 / Roboflow Universe) and swap `data=` to that.

In [ ]:
# Fine-tune the detection variant (boxes + labels)
# det_model = YOLO("yolo11s.pt")
# det_model.train(data=str(yaml_det), epochs=60, imgsz=640, batch=16, device=device,
#                 project="runs/yolo11_food100", name="det_train", exist_ok=True)

## 9. Next step: plug into Nutrify

The exported model (or `best.pt`) can replace/augment the current pipeline in `main.py`:

```
POST /predict        → YOLO classify uploaded photo → nutrition lookup (same as now)
WS  /predict-frame   → live frames from the phone/desktop camera → label overlay
```

Class names in the app's `food_nutrition` table must match the folder names used above (e.g., `chicken_wings`).

After export, copy `realtime_food_recognition_100_foods.pt`, `realtime_food_recognition_100_foods.onnx`, and `realtime_food_recognition_100_foods_classes.txt` from the Drive `deploy/` folder into this project's `model/` folder.